# Article 1 — ungated Grounded SAM 2 runner

This notebook uses **Grounding DINO 1.0** to find text-prompted objects in the first frame and **SAM 2.1** to segment and track them through the video. Both checkpoints are downloaded from their projects' public release URLs; no Hugging Face approval or model token is required.

Before running: select a **GPU** runtime. The default smoke test uses the same public bedroom MP4, `person` prompt, 3 seconds, 4 fps, and 560-pixel limit as the SAM3 notebook. While the article repository is private, add a read-only `GITHUB_TOKEN` under Colab **Secrets**.

In [ ]:
import platform
import subprocess
import sys

print(f"Python: {platform.python_version()}")
assert sys.version_info >= (3, 12), "The article package requires Python 3.12+"
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from base64 import b64encode
from pathlib import Path
import time
from urllib.request import urlretrieve

from google.colab import userdata

ROOT = Path("/content")
GVI = ROOT / "grounded-visual-intelligence"
GROUNDED_SAM2 = ROOT / "Grounded-SAM-2"

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Add a read-only GITHUB_TOKEN while the article repo is private")
basic = b64encode(f"x-access-token:{github_token}".encode()).decode()

if not GVI.exists():
    subprocess.run(
        [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {basic}",
            "clone",
            "--branch",
            "article/01-grounded-video-memory",
            "--depth",
            "1",
            "https://github.com/vlada22/grounded-visual-intelligence.git",
            str(GVI),
        ],
        check=True,
    )
if not GROUNDED_SAM2.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/IDEA-Research/Grounded-SAM-2.git",
            str(GROUNDED_SAM2),
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(GVI),
        "-e",
        str(GROUNDED_SAM2),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-build-isolation",
        "-e",
        str(GROUNDED_SAM2 / "grounding_dino"),
    ],
    check=True,
)

CHECKPOINTS = ROOT / "gvi-checkpoints"
CHECKPOINTS.mkdir(exist_ok=True)
SAM2_CHECKPOINT = CHECKPOINTS / "sam2.1_hiera_large.pt"
GDINO_CHECKPOINT = CHECKPOINTS / "groundingdino_swint_ogc.pth"
checkpoint_download_started = time.perf_counter()
if not SAM2_CHECKPOINT.exists():
    urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
        SAM2_CHECKPOINT,
    )
if not GDINO_CHECKPOINT.exists():
    urlretrieve(
        "https://github.com/IDEA-Research/GroundingDINO/releases/download/"
        "v0.1.0-alpha/groundingdino_swint_ogc.pth",
        GDINO_CHECKPOINT,
    )
checkpoint_download_s = time.perf_counter() - checkpoint_download_started
source_dirs = [GVI / "src", GROUNDED_SAM2, GROUNDED_SAM2 / "grounding_dino"]
for source_dir in source_dirs:
    source_dir = str(source_dir)
    if source_dir not in sys.path:
        sys.path.insert(0, source_dir)

from gvi.adapters.grounded_sam2 import GroundedObjectSeed
from gvi.run_artifacts import sha256_file

sam2_checkpoint_sha256 = sha256_file(SAM2_CHECKPOINT)
gdino_checkpoint_sha256 = sha256_file(GDINO_CHECKPOINT)
grounded_sam2_revision = subprocess.run(
    ["git", "-C", str(GROUNDED_SAM2), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(
    {
        "grounded_sam2_revision": grounded_sam2_revision,
        "sam2_checkpoint_sha256": sam2_checkpoint_sha256,
        "gdino_checkpoint_sha256": gdino_checkpoint_sha256,
    }
)

In [ ]:
from google.colab import files
from huggingface_hub import hf_hub_download

USE_OFFICIAL_SAMPLE = True
PROMPT = "person."  # Grounding DINO prompts conventionally end with a period.
SOURCE_URI = "hf://datasets/hf-internal-testing/sam2-fixtures/bedroom.mp4"

if USE_OFFICIAL_SAMPLE:
    source_download_started = time.perf_counter()
    original_video_path = Path(
        hf_hub_download(
            repo_id="hf-internal-testing/sam2-fixtures",
            repo_type="dataset",
            filename="bedroom.mp4",
        )
    )
    source_download_s = time.perf_counter() - source_download_started
else:
    print("Upload the identical MP4 used by the SAM3 notebook.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one video")
    original_video_path = ROOT / next(iter(uploaded))
    PROMPT = "red toy car."
    SOURCE_URI = f"uploaded://{original_video_path.name}"
    source_download_s = 0.0

video_path = ROOT / "article-01-comparison.mp4"
preprocess_started = time.perf_counter()
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        str(original_video_path),
        "-t",
        "3",
        "-vf",
        "fps=4,scale='min(560,iw)':-2:flags=lanczos",
        "-an",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        str(video_path),
    ],
    check=True,
)
preprocess_s = time.perf_counter() - preprocess_started
source_sha256 = sha256_file(original_video_path)
prepared_sha256 = sha256_file(video_path)
print(
    {
        "source": original_video_path.name,
        "prepared": video_path.name,
        "prompt": PROMPT,
        "source_sha256": source_sha256,
        "prepared_sha256": prepared_sha256,
    }
)


In [ ]:
import cv2

FRAME_DIR = ROOT / "article-01-frames"
FRAME_DIR.mkdir(exist_ok=True)
capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise RuntimeError(f"Could not open {video_path}")
fps = float(capture.get(cv2.CAP_PROP_FPS))
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = 0
while True:
    ok, frame = capture.read()
    if not ok:
        break
    cv2.imwrite(str(FRAME_DIR / f"{frame_count:05d}.jpg"), frame)
    frame_count += 1
capture.release()
if fps <= 0 or frame_count == 0:
    raise RuntimeError("Video metadata is invalid")
{"width": width, "height": height, "fps": fps, "frame_count": frame_count}

In [ ]:
import time

import numpy as np
import torch
from grounding_dino.groundingdino.util.inference import load_image, load_model, predict
from sam2.build_sam import build_sam2_video_predictor
from torchvision.ops import box_convert

from gvi.adapters.grounded_sam2 import GroundedObjectSeed
from gvi.models import BoundingBox

BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25
PROMPT_FRAME_INDEX = 0

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model_load_started = time.perf_counter()
grounding_model = load_model(
    str(GROUNDED_SAM2 / "grounding_dino/groundingdino/config/GroundingDINO_SwinT_OGC.py"),
    str(GDINO_CHECKPOINT),
    device="cuda",
)
video_predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    str(SAM2_CHECKPOINT),
)
torch.cuda.synchronize()
model_load_s = time.perf_counter() - model_load_started

prompt_frame = FRAME_DIR / f"{PROMPT_FRAME_INDEX:05d}.jpg"
image_source, image = load_image(str(prompt_frame))
torch.cuda.synchronize()
grounding_started = time.perf_counter()
boxes, confidences, labels = predict(
    model=grounding_model,
    image=image,
    caption=PROMPT,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
)
torch.cuda.synchronize()
grounding_s = time.perf_counter() - grounding_started
if len(boxes) == 0:
    raise RuntimeError("Grounding DINO found no objects; inspect the prompt or threshold")
scale = torch.tensor([width, height, width, height], device=boxes.device)
input_boxes = box_convert(boxes * scale, in_fmt="cxcywh", out_fmt="xyxy").cpu().numpy()
scores = confidences.cpu().numpy().tolist()
object_ids = list(range(1, len(labels) + 1))
seeds = tuple(
    GroundedObjectSeed(
        object_id=object_id,
        label=label,
        grounding_score=score,
        prompt_frame_index=PROMPT_FRAME_INDEX,
        initial_box=BoundingBox(
            x_min=float(box[0]),
            y_min=float(box[1]),
            x_max=float(box[2]),
            y_max=float(box[3]),
        ),
    )
    for object_id, label, score, box in zip(object_ids, labels, scores, input_boxes, strict=True)
)
[(seed.object_id, seed.label, round(seed.grounding_score, 3)) for seed in seeds]

In [ ]:
import json

artifact_export_torch.cuda.synchronize()
started = time.perf_counter()
from gvi.adapters.grounded_sam2 import (
    GroundedSam2Adapter,
    GroundedSam2FrameRecord,
    GroundedSam2RecordedOutput,
)
from gvi.inference.mask_rle import encode_binary_mask

SCENE_ID = "article-01-hero-grounded-sam2"
OUTPUT_DIR = ROOT / "gvi-artifacts" / SCENE_ID
MASK_DIR = OUTPUT_DIR / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

session_started = time.perf_counter()
state = video_predictor.init_state(video_path=str(FRAME_DIR))
session_init_s = time.perf_counter() - session_started
amp_dtype = torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
frame_records = []
with torch.inference_mode(), torch.autocast("cuda", dtype=amp_dtype):
    for object_id, box in zip(object_ids, input_boxes, strict=True):
        video_predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=PROMPT_FRAME_INDEX,
            obj_id=object_id,
            box=box,
        )

    for frame_index, propagated_ids, mask_logits in video_predictor.propagate_in_video(state):
        frame_object_ids = []
        frame_boxes = []
        frame_areas = []
        frame_mask_uris = []
        for index, object_id in enumerate(propagated_ids):
            mask = (mask_logits[index] > 0).detach().cpu().numpy().squeeze()
            rows, columns = np.nonzero(mask)
            if len(rows) == 0:
                continue
            object_id = int(object_id)
            relative_mask = Path("masks") / f"frame-{frame_index:05d}-object-{object_id}.rle.json"
            (OUTPUT_DIR / relative_mask).write_text(
                encode_binary_mask(mask).model_dump_json(indent=2),
                encoding="utf-8",
            )
            frame_object_ids.append(object_id)
            frame_boxes.append(
                BoundingBox(
                    x_min=float(columns.min()),
                    y_min=float(rows.min()),
                    x_max=float(columns.max() + 1),
                    y_max=float(rows.max() + 1),
                )
            )
            frame_areas.append(int(mask.sum()))
            frame_mask_uris.append(relative_mask.as_posix())
        frame_records.append(
            GroundedSam2FrameRecord(
                frame_index=int(frame_index),
                object_ids=tuple(frame_object_ids),
                boxes_xyxy=tuple(frame_boxes),
                mask_areas=tuple(frame_areas),
                mask_uris=tuple(frame_mask_uris),
            )
        )
torch.cuda.synchronize()
inference_s = time.perf_counter() - started

recorded = GroundedSam2RecordedOutput(
    scene_id=SCENE_ID,
    source_uri=SOURCE_URI,
    prompt=PROMPT.rstrip("."),
    width=width,
    height=height,
    fps=fps,
    frame_count=frame_count,
    seeds=seeds,
    frames=tuple(frame_records),
)
(OUTPUT_DIR / "grounded-sam2-output.json").write_text(
    recorded.model_dump_json(indent=2), encoding="utf-8"
)
scene = GroundedSam2Adapter().convert(recorded)
(OUTPUT_DIR / "scene.json").write_text(scene.model_dump_json(indent=2), encoding="utf-8")
artifact_export_s = time.perf_counter() - artifact_export_started
metrics = {
    "pipeline": "grounding-dino-1.0+sam-2.1",
    "runtime": "colab",
    "runtime_metadata": None,
    "grounded_sam2_revision": grounded_sam2_revision,
    "sam2_checkpoint_sha256": sam2_checkpoint_sha256,
    "gdino_checkpoint_sha256": gdino_checkpoint_sha256,
    "source_uri": SOURCE_URI,
    "source_sha256": source_sha256,
    "prepared_sha256": prepared_sha256,
    "checkpoint_download_s": checkpoint_download_s,
    "source_download_s": source_download_s,
    "preprocess_s": preprocess_s,
    "session_init_s": session_init_s,
    "artifact_export_s": artifact_export_s,
    "prompt": PROMPT.rstrip("."),
    "comparison_preset": {"seconds": 3, "fps": 4, "max_width": 560},
    "model_load_s": model_load_s,
    "grounding_s": grounding_s,
    "inference_s": inference_s,
    "total_model_s": model_load_s + grounding_s + session_init_s + inference_s,
    "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "tracked_frames": len(frame_records),
    "source_frames": frame_count,
}
metrics

In [ ]:
import shutil

from gvi.run_artifacts import build_analysis_artifacts, runtime_metadata, write_manifest

analysis_started = time.perf_counter()
analysis_summary = build_analysis_artifacts(
    output_dir=OUTPUT_DIR,
    video_path=video_path,
    scene_path=OUTPUT_DIR / "scene.json",
)
analysis_artifact_s = time.perf_counter() - analysis_started
metrics["analysis_artifact_s"] = analysis_artifact_s
metrics["runtime_metadata"] = runtime_metadata(torch)
metrics["peak_gpu_memory_allocated_gib"] = torch.cuda.max_memory_allocated() / 1024**3
metrics["peak_gpu_memory_reserved_gib"] = torch.cuda.max_memory_reserved() / 1024**3
metrics["analysis_summary"] = analysis_summary
(OUTPUT_DIR / "run-metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
manifest = write_manifest(OUTPUT_DIR)

archive_started = time.perf_counter()
archive = shutil.make_archive(str(ROOT / SCENE_ID), "zip", OUTPUT_DIR)
archive_s = time.perf_counter() - archive_started
print(
    {
        "artifact_bundle": archive,
        "archive_s": archive_s,
        "files": len(manifest["files"]),
        "archive_sha256": sha256_file(Path(archive)),
    }
)
files.download(archive)


## Expected output

The ZIP contains the prepared comparison clip, annotated preview video, contact
sheet, per-frame track measurements, analysis summary, integrity manifest, raw
Grounded SAM 2 recording, model-independent `scene.json`, RLE masks, checkpoint
and source hashes, and detailed phase/runtime metrics.
